In [ ]:
import math
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import seaborn as sns

CACHE_PATH = Path("data/cache/qwen25_gqa_cache.pt")
cache = torch.load(CACHE_PATH, map_location="cpu")

meta = cache["meta"]
prompts = cache["prompts"]

print("Model:", cache["model_id"])
print("Q heads:", meta["num_q_heads"], "| KV heads:", meta["num_kv_heads"], "| KV groups:", meta["num_kv_groups"])

PROMPT_IDX = 0
LAYER_IDX = 0
MAX_LABEL_LEN = 18
FIGSIZE_PER_PANEL = 3.2

record = prompts[PROMPT_IDX]
tokens = record["tokens"]
attn = record["attentions"][LAYER_IDX]   # [bsz, q_heads, tgt_len, src_len]
assert attn is not None, "Attention weights are missing. Ensure attn_implementation='eager'."

attn = attn[0].float()                   # [q_heads, seq_len, seq_len]
num_heads, tgt_len, src_len = attn.shape
kv_map = record["layers"][LAYER_IDX]["kv_head_for_q"].tolist()

def short_tok(t: str, max_len: int = MAX_LABEL_LEN) -> str:
    t = t.replace("Ġ", "▁").replace("Ċ", "\\n")
    return t if len(t) <= max_len else t[: max_len - 1] + "…"

labels = [short_tok(t) for t in tokens]

ncols = 4
nrows = math.ceil(num_heads / ncols)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * FIGSIZE_PER_PANEL, nrows * FIGSIZE_PER_PANEL),
    squeeze=False
)

for h in range(nrows * ncols):
    ax = axes[h // ncols][h % ncols]
    if h >= num_heads:
        ax.axis("off")
        continue

    sns.heatmap(
        attn[h].numpy(),
        ax=ax,
        cmap="magma",
        vmin=0.0,
        vmax=float(attn.max()),
        cbar=False,
        square=True,
        xticklabels=labels,
        yticklabels=labels,
    )
    ax.set_title(f"Head q={h} → kv={kv_map[h]}")
    ax.set_xlabel("Source token")
    ax.set_ylabel("Target token")
    ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

fig.suptitle(
    f"Attention heatmaps | prompt={PROMPT_IDX} | layer={LAYER_IDX}\n"
    f"{record['text']}",
    y=1.02
)
plt.tight_layout()
plt.show()